# SFDC ↔ Airtable Job Reconciliation

Reconstructs the AT-vs-SFDC reconciliation table (the Google-Sheet exercise) directly from
the **Airtable API** + a **Salesforce report CSV**, and emits a row-level list of the jobs
whose attributes disagree between the two systems.

**Inputs**
1. A Salesforce report CSV in the format you paste in (one row per *Site Service*).
   Drop new versions into `data/` and point `SFDC_CSV_PATH` at the latest.
2. The Airtable **Job Tracking** table (base `Command Context Sync`), pulled live via API.

**Outputs**
1. `reconciliation_summary` — the AT / SFDC side-by-side metric table (Customers, Locations,
   Rows/SS, Jobs, Sqft) for every Status × Filter × Customer × Type combination in the sheet,
   plus the *Adjusted-for-Prime* bridge and the **Delta** row.
2. `job_diffs.md` — a bulleted list of every job whose fields differ, in the form:

   ```
   * <site>
       * <attribute>: X on SFDC, Y on AT
   ```

**Join key:** SFDC `18char Job ID` (e.g. `a0RVs0000094JZzMAM`) ⇄ Airtable
`18char Job ID Rollup (from new_sfdc_job_sync)`.

> The numbers will drift slightly from any given screenshot because the CSV and the Airtable
> pull are taken at different moments — that drift *is* the thing we are measuring.


## 1. Configuration

The Airtable token is read from the environment (`AIRTABLE_API_TOKEN`, falling back to
`AIRTABLE_API_KEY` / `AIRTABLE_TOKEN`). Set it before launching Jupyter, e.g.

```bash
export AIRTABLE_API_TOKEN="patXXXXXXXX..."
```

All base/table/field IDs below were resolved from the live schema, so nothing here relies on
display names that someone might rename in the Airtable UI.

In [ ]:
import os, re, json, time, datetime as dt
from collections import defaultdict
import requests
import pandas as pd

# ---- Airtable identifiers (resolved from the live schema) ----------------------------------
AIRTABLE_API_TOKEN = (
    os.environ.get("AIRTABLE_API_TOKEN")
    or os.environ.get("AIRTABLE_API_KEY")
    or os.environ.get("AIRTABLE_TOKEN")
)
BASE_ID  = "app7jMwevErzRNk7G"      # Command Context Sync
TABLE_ID = "tblv8G1wvbMyzJpJd"      # Job Tracking

# Airtable field IDs we need (id -> friendly name). Using IDs survives field renames.
AT_FIELDS = {
    "job_id":        "fld8yRsobul4XnOuN",  # 18char Job ID Rollup (from new_sfdc_job_sync)  <-- JOIN KEY
    "account":       "fldXxKR4zpyjpgEba",  # new_sfdc_account_name
    "address":       "fldtw7MdvgzEDCgFd",  # new_sfdc_concat_full_address
    "sqft":          "fld5s9VX2hp8CLr27",  # Actual Square Footage Rollup (from new_sfdc_ol_sync)
    "total_sqft":    "fld3rjmwpIisZncjZ",  # Total Sq. Ft.
    "job_status":    "fldVkbhy7xLFLWGNN",  # Job Status
    "job_type":      "fldEaFaSjZfq1BEWe",  # Job Type
    "network_live":  "fld6ehsRla0PEYvUQ",  # Network Live? (Jobs)  (checkbox)
    "target_golive": "fldujSlIehKhZXvbN",  # Target Go-Live Date (Meter)
    "actual_golive": "fldDvLA5uaH5gWa2J",  # Actual Go-Live Date
    "is_prime":      "fld9hprabcxAZ9IWX",  # is_prime  (1 = Prime Group Holdings)
    "connect_cust":  "fldwcdAQtzdGu8TiL",  # Connect customer?
    "job_name":      "fldWZE2JbBTALWlmg",  # Job ID  (JOB-xxxx label, for readability)
}

# ---- SFDC report CSV -----------------------------------------------------------------------
SFDC_CSV_PATH = "data/sfdc_report_2026-06-22.csv"   # <-- point at the latest export
SFDC_CSV_ENCODING = "latin-1"                       # the SFDC export is not clean utf-8

# SFDC column names (left) -> canonical name (right)
SFDC_COLS = {
    "18char Job ID":                    "job_id",
    "Job: Account Name":                "account",
    "OL: Full Address":                 "address",
    "Actual Square Footage":            "sqft",
    "Job: Job Status":                  "job_status",
    "Job: Job Type":                    "job_type",
    "Job: Network Live?":               "network_live",
    "Job: Target Go-Live Date (Meter)": "target_golive",
    "Actual Go-Live Date":              "actual_golive",
    "Site Service: Site Service Name":  "site_service",
    "Job Name":                         "job_name",
    "ProvSite: Opportunity Name":       "opportunity",
    "Cellular + Connect Only":          "cellular_connect",
}

# ---- Business rules (edit here if the report definition changes) ---------------------------
# "Focused" filter set, minus the dimensions broken out as their own columns (Status / Type).
EXCLUDED_ACCOUNTS   = {"Amrize", "Meter", "Lineage Logistics"}
EXCLUDED_STATUSES   = {"Complete", "Canceled", "Cancelled"}
# Job Types removed by the full "Focused" definition:
NON_FOCUSED_TYPES   = {"Pre-install", "Connect-only", "Cellular-only", "NFR"}
# "Adjusted" type column == everything except Pre-install:
PREINSTALL_TYPES    = {"Pre-install"}
PRIME_ACCOUNTS      = {"Prime Group Holdings LLC"}  # the "Prime" carved out of the bridge

assert AIRTABLE_API_TOKEN, "Set AIRTABLE_API_TOKEN in the environment before running."
print("Config loaded. CSV:", SFDC_CSV_PATH)

## 2. Pull the Airtable Job Tracking table

Straight REST (`GET /v0/{base}/{table}`) with cursor pagination, requesting only the fields we
need. ~4,900 records, so this is a handful of pages.

In [ ]:
def fetch_airtable_records(base_id, table_id, field_ids, token, page_size=100):
    """Yield every record from a table, requesting only `field_ids`."""
    url = f"https://api.airtable.com/v0/{base_id}/{table_id}"
    headers = {"Authorization": f"Bearer {token}"}
    params = [("pageSize", page_size), ("returnFieldsByFieldId", "true")]
    params += [("fields[]", fid) for fid in field_ids]
    offset = None
    while True:
        p = list(params) + ([("offset", offset)] if offset else [])
        for attempt in range(5):
            r = requests.get(url, headers=headers, params=p, timeout=60)
            if r.status_code == 429:          # rate limited
                time.sleep(2 ** attempt)
                continue
            r.raise_for_status()
            break
        data = r.json()
        for rec in data["records"]:
            yield rec
        offset = data.get("offset")
        if not offset:
            break

def _scalar(v):
    """Flatten Airtable cell values to plain scalars."""
    if isinstance(v, dict):                      # singleSelect / collaborator
        return v.get("name", v.get("id"))
    if isinstance(v, list):                      # multiSelect / rollup / link
        if not v:
            return None
        return ", ".join(_scalar(x) for x in v if x is not None)
    return v

raw = list(fetch_airtable_records(BASE_ID, TABLE_ID, list(AT_FIELDS.values()), AIRTABLE_API_TOKEN))
print(f"Pulled {len(raw)} Airtable records")

rows = []
inv = {fid: name for name, fid in AT_FIELDS.items()}
for rec in raw:
    f = rec.get("fields", {})
    row = {"at_record_id": rec["id"]}
    for fid, name in inv.items():
        row[name] = _scalar(f.get(fid))
    rows.append(row)
at = pd.DataFrame(rows)
at.head(3)

## 3. Load & normalise both sides

We coerce both frames to a shared canonical schema so every downstream comparison is
apples-to-apples. The join key is normalised (trimmed, upper-cased).

In [ ]:
def norm_id(x):
    return str(x).strip().upper() if pd.notna(x) and str(x).strip() else None

def norm_text(x):
    if pd.isna(x):
        return ""
    return re.sub(r"\s+", " ", str(x)).strip()

def to_float(x):
    if pd.isna(x) or str(x).strip() == "":
        return 0.0
    try:
        return float(str(x).replace(",", ""))
    except ValueError:
        return 0.0

def to_bool_live(x):
    # SFDC exports "0"/"1"; Airtable checkbox -> True/None
    if isinstance(x, bool):
        return x
    return str(x).strip() in {"1", "True", "true", "Yes"}

# ---- SFDC ----------------------------------------------------------------------------------
sfdc_raw = pd.read_csv(SFDC_CSV_PATH, dtype=str, encoding=SFDC_CSV_ENCODING).fillna("")
sfdc = sfdc_raw.rename(columns=SFDC_COLS)[list(SFDC_COLS.values())].copy()
sfdc["job_id_key"]    = sfdc["job_id"].map(norm_id)
sfdc["network_live"]  = sfdc["network_live"].map(to_bool_live)
sfdc["sqft_num"]      = sfdc["sqft"].map(to_float)
sfdc["cellular_connect_num"] = sfdc["cellular_connect"].map(to_float)
sfdc["source"] = "SFDC"

# ---- Airtable ------------------------------------------------------------------------------
at["job_id_key"]   = at["job_id"].map(norm_id)
at["network_live"] = at["network_live"].map(to_bool_live)
at["sqft_num"]     = at["sqft"].map(to_float)
at["is_prime_num"] = at["is_prime"].map(to_float)
at["site_service"] = ""          # AT grain is the site-service row already
at["cellular_connect_num"] = at["connect_cust"].map(lambda v: 1.0 if norm_text(v) else 0.0)
at["source"] = "AT"

print("SFDC rows:", len(sfdc), "| Airtable rows:", len(at))
print("SFDC rows with a job id:", sfdc["job_id_key"].notna().sum())
print("AT  rows with a job id:", at["job_id_key"].notna().sum())

## 4. Filter predicates

Each dimension in the reconciliation sheet is a predicate. They compose, so any
Status × Filter × Customer × Type combination from the screenshots can be requested by name.

| Dimension | Values | Rule |
|---|---|---|
| **Status** | `Live` / `Pre-Live` | `Network Live?` == True / False |
| **Filters** | `None` / `Focused` | Focused ⇒ drop excluded accounts, excluded statuses, cellular+connect, non-focused job types |
| **Customer** | `All` / a specific account | exact account-name match |
| **Type** | `All` / `Adjusted` / `Pre-Install` | Adjusted ⇒ Job Type ≠ Pre-install; Pre-Install ⇒ Job Type == Pre-install |


In [ ]:
def predicate(df, *, status=None, filters="None", customer="All", type_="All",
              drop_prime=False):
    m = pd.Series(True, index=df.index)

    if status == "Live":
        m &= df["network_live"] == True
    elif status == "Pre-Live":
        m &= df["network_live"] == False

    if filters == "Focused":
        m &= ~df["account"].isin(EXCLUDED_ACCOUNTS)
        m &= ~df["job_status"].isin(EXCLUDED_STATUSES)
        m &= df["cellular_connect_num"] != 1
        m &= ~df["job_type"].isin(NON_FOCUSED_TYPES)

    if customer != "All":
        m &= df["account"] == customer

    if type_ == "Adjusted":
        m &= ~df["job_type"].isin(PREINSTALL_TYPES)
    elif type_ == "Pre-Install":
        m &= df["job_type"].isin(PREINSTALL_TYPES)

    if drop_prime:
        m &= ~df["account"].isin(PRIME_ACCOUNTS)

    return df[m]

def metrics(df):
    """The five reconciliation measures for a filtered frame."""
    return {
        "Customers": df["account"].replace("", pd.NA).nunique(),
        "Locations": df["address"].map(norm_text).replace("", pd.NA).nunique(),
        "Rows":      len(df),
        "Jobs":      df["job_id_key"].nunique(),
        "Sqft":      int(df["sqft_num"].sum()),
    }

## 5. Reconstruct the reconciliation summary

This reproduces the row layout of the AIRTABLE / SFDC blocks in the sheet. Edit `COMBINATIONS`
to add or drop rows.

In [ ]:
COMBINATIONS = [
    # label,                              kwargs
    ("Live  | None     | All",            dict(status="Live",     filters="None")),
    ("PreLv | None     | All",            dict(status="Pre-Live", filters="None")),
    ("Live  | Focused  | All",            dict(status="Live",     filters="Focused")),
    ("PreLv | Focused  | All",            dict(status="Pre-Live", filters="Focused")),
    ("Live  | Focused  | Adjusted",       dict(status="Live",     filters="Focused", type_="Adjusted")),
    ("PreLv | Focused  | Adjusted",       dict(status="Pre-Live", filters="Focused", type_="Adjusted")),
    ("Live  | Focused  | Pre-Install",    dict(status="Live",     filters="Focused", type_="Pre-Install")),
    ("PreLv | Focused  | Pre-Install",    dict(status="Pre-Live", filters="Focused", type_="Pre-Install")),
    ("Live  | All      | Pre-Install",    dict(status="Live",     filters="None",    type_="Pre-Install")),
    ("PreLv | All      | Pre-Install",    dict(status="Pre-Live", filters="None",    type_="Pre-Install")),
]

def build_summary(combinations):
    out = []
    for label, kw in combinations:
        a = metrics(predicate(at, **kw))
        s = metrics(predicate(sfdc, **kw))
        row = {"Combination": label}
        for k in ["Customers", "Locations", "Rows", "Jobs", "Sqft"]:
            row[f"AT.{k}"]    = a[k]
            row[f"SFDC.{k}"]  = s[k]
            row[f"Δ.{k}"]     = a[k] - s[k]
        out.append(row)
    return pd.DataFrame(out)

reconciliation_summary = build_summary(COMBINATIONS)
pd.set_option("display.max_columns", None, "display.width", 200)
reconciliation_summary

### 5b. The "Adjusted for Prime" bridge + Delta

The orange *Comparison* block: Pre-Live, Focused, Adjusted, **Prime Group Holdings removed**,
so the two systems can be lined up without Prime's large-but-volatile footprint skewing it.

In [ ]:
bridge_kw = dict(status="Pre-Live", filters="Focused", type_="Adjusted", drop_prime=True)
at_bridge   = metrics(predicate(at,   **bridge_kw))
sfdc_bridge = metrics(predicate(sfdc, **bridge_kw))

bridge = pd.DataFrame(
    [at_bridge, sfdc_bridge, {k: at_bridge[k] - sfdc_bridge[k] for k in at_bridge}],
    index=["AT Adj. for Prime", "SFDC Adj. for Prime", "Delta (AT - SFDC)"],
)
bridge

## 6. Row-level diff — which jobs disagree, and on what

Inner-join on the job id; for every shared job compare each attribute. (When several rows share
a job id we compare the aggregated/first value per job.) The result is written to `job_diffs.md`
in the requested bullet format.

In [ ]:
# Attributes to compare, and how to render each value.
COMPARE_ATTRS = {
    "account":       ("Account",        norm_text),
    "address":       ("Address",        norm_text),
    "sqft":          ("Sq Ft",          lambda v: f"{to_float(v):,.0f}"),
    "job_status":    ("Job Status",     norm_text),
    "job_type":      ("Job Type",       norm_text),
    "target_golive": ("Target Go-Live", norm_text),
    "actual_golive": ("Actual Go-Live", norm_text),
    "network_live":  ("Network Live?",  lambda v: str(bool(v))),
}

def collapse(df, key="job_id_key"):
    """One row per job id (first non-empty value per attribute)."""
    def first_nonempty(s):
        for v in s:
            if pd.notna(v) and str(v).strip() not in ("", "nan"):
                return v
        return s.iloc[0] if len(s) else None
    cols = [c for c in COMPARE_ATTRS if c in df.columns] + ["site_service", "job_name"]
    cols = [c for c in cols if c in df.columns]
    return df.dropna(subset=[key]).groupby(key)[cols].agg(first_nonempty)

at_by_job   = collapse(at)
sfdc_by_job = collapse(sfdc)

shared = sorted(set(at_by_job.index) & set(sfdc_by_job.index))
only_sfdc = sorted(set(sfdc_by_job.index) - set(at_by_job.index))
only_at   = sorted(set(at_by_job.index)   - set(sfdc_by_job.index))
print(f"Shared jobs: {len(shared)} | only in SFDC: {len(only_sfdc)} | only in AT: {len(only_at)}")

def render_val(attr, v):
    fmt = COMPARE_ATTRS[attr][1]
    try:
        return fmt(v)
    except Exception:
        return norm_text(v)

diffs = []          # (site_label, [(attr_label, sfdc_val, at_val), ...])
for jid in shared:
    s, a = sfdc_by_job.loc[jid], at_by_job.loc[jid]
    site = (str(s.get("site_service") or "").strip()
            or str(s.get("job_name") or "").strip()
            or jid)
    addr = norm_text(s.get("address"))
    site_label = f"{site} — {addr}" if addr else site
    rowdiffs = []
    for attr, (label, _) in COMPARE_ATTRS.items():
        if attr == "sqft":
            if abs(to_float(s.get("sqft")) - to_float(a.get("sqft"))) > 0.5:
                rowdiffs.append((label, render_val(attr, s.get("sqft")), render_val(attr, a.get("sqft"))))
            continue
        sv, av = render_val(attr, s.get(attr)), render_val(attr, a.get(attr))
        if sv != av:
            rowdiffs.append((label, sv, av))
    if rowdiffs:
        diffs.append((site_label, jid, rowdiffs))

print(f"Jobs with at least one differing attribute: {len(diffs)}")

In [ ]:
# Build the markdown bullet list in the requested shape.
lines = []
lines.append(f"# SFDC ↔ AT job diffs  ({dt.date.today().isoformat()})\n")
lines.append(f"- Shared jobs compared: **{len(shared)}**")
lines.append(f"- Jobs with differing attributes: **{len(diffs)}**")
lines.append(f"- Jobs only in SFDC: **{len(only_sfdc)}**  |  only in Airtable: **{len(only_at)}**\n")

lines.append("## Attribute differences\n")
for site_label, jid, rowdiffs in diffs:
    lines.append(f"* {site_label}  `({jid})`")
    for label, sv, av in rowdiffs:
        lines.append(f"   * {label}: {sv} on SFDC, {av} on AT")

if only_sfdc:
    lines.append("\n## Jobs present in SFDC but missing from Airtable\n")
    for jid in only_sfdc:
        s = sfdc_by_job.loc[jid]
        site = str(s.get("site_service") or s.get("job_name") or "").strip()
        lines.append(f"* {site} `({jid})` — {norm_text(s.get('address'))}")

if only_at:
    lines.append("\n## Jobs present in Airtable but missing from SFDC\n")
    for jid in only_at:
        a = at_by_job.loc[jid]
        lines.append(f"* {str(a.get('job_name') or '').strip()} `({jid})` — {norm_text(a.get('address'))}")

md_text = "\n".join(lines)
with open("job_diffs.md", "w") as fh:
    fh.write(md_text)
print("Wrote job_diffs.md")
print("\n".join(md_text.splitlines()[:40]))

## 7. (Optional) Attribute changes over time — the audit log

Everything above is a **point-in-time** snapshot: "right now, these N jobs disagree." It tells
you *that* they differ, not *which side moved, when, or who moved it*.

Your token has access to the Airtable **audit log** (Enterprise admin API,
`/v0/meta/enterpriseAccounts/{accountId}/auditLogEvents`). For the handful of jobs flagged
above you can pull the recent change events on the differing fields to see whether Airtable was
edited after the last SFDC sync — i.e. *which system drifted*. Fill in your enterprise account
id to use it.

In [ ]:
ENTERPRISE_ACCOUNT_ID = os.environ.get("AIRTABLE_ENTERPRISE_ACCOUNT_ID")  # entXXXXXXXX

def fetch_audit_events(account_id, token, *, start_time=None, page_limit=5):
    """Page the enterprise audit log (most-recent first). Returns raw event dicts."""
    url = f"https://api.airtable.com/v0/meta/enterpriseAccounts/{account_id}/auditLogEvents"
    headers = {"Authorization": f"Bearer {token}"}
    params = {"pageSize": 100, "sortOrder": "descending"}
    if start_time:
        params["startTime"] = start_time
    events, pages = [], 0
    while pages < page_limit:
        r = requests.get(url, headers=headers, params=params, timeout=60)
        r.raise_for_status()
        data = r.json()
        events.extend(data.get("events", []))
        nxt = data.get("pagination", {}).get("next")
        if not nxt:
            break
        params["next"] = nxt
        pages += 1
    return events

if ENTERPRISE_ACCOUNT_ID:
    since = (dt.datetime.utcnow() - dt.timedelta(days=14)).strftime("%Y-%m-%dT%H:%M:%SZ")
    evts = fetch_audit_events(ENTERPRISE_ACCOUNT_ID, AIRTABLE_API_TOKEN, start_time=since)
    print(f"Pulled {len(evts)} audit events in the last 14 days")
    # Keep only record-update events on the Job Tracking table that touch our diffing records.
    diff_record_ids = set(at[at["job_id_key"].isin([j for _, j, _ in diffs])]["at_record_id"])
    relevant = [e for e in evts
                if e.get("action") in {"updateRecord", "updateRecords"}
                and any(rid in json.dumps(e) for rid in diff_record_ids)]
    print(f"  ...of which {len(relevant)} touch a currently-diffing record")
else:
    print("Set AIRTABLE_ENTERPRISE_ACCOUNT_ID to enable audit-log attribution.")

## Do we need to scan this over time?

**Short answer: not for the first pass. Run it point-in-time, but snapshot the output so you
can turn on trend tracking later without re-plumbing anything.**

Reasoning:

- **The reconciliation itself is a snapshot.** "These 12 jobs disagree today" is the actionable
  deliverable. You fix them, re-run, and the list shrinks. You don't need history to *act*.
- **A lot of today's delta is just sync lag, not real divergence.** SFDC is exported manually;
  Airtable updates continuously. Many diffs resolve themselves on the next sync. Time-scanning is
  what lets you *distinguish* "transient lag" from "genuine, sticky drift" — but you only need
  that once the same job keeps reappearing on the list.
- **When you *do* want over-time, you want two different things, and only one needs the audit log:**
  1. *Trend of the gap* — is the total delta shrinking or growing? → cheap: append each run's
     `reconciliation_summary` to a dated log (one row per run). Section 5 already produces it.
  2. *Attribution of a specific diff* — who/what changed `Job Status` on this job, and was it
     after the last SFDC pull? → that needs the **audit log** (Section 7), and only for the small
     set of jobs currently flagged. Don't scan the whole base's history; scan the diffs.

**Recommendation**

1. Keep running this notebook per CSV drop (point-in-time).
2. Append `reconciliation_summary` + the diff count to a `runs/` log each time (timestamped) —
   that gives you the trend line for free, no audit log required.
3. Only reach for the audit log to *attribute* a diff that won't go away — i.e. a job that shows
   up on the list two runs in a row. That's where "scan over time" earns its keep.

So: **point-in-time now; persist the summaries so over-time is a switch you can flip, not a
rebuild; and reserve the audit-log scan for sticky diffs rather than running it across the whole
base continuously.**
